# Clustering of head-direction tuning: neurons and subjects

Four questions, all using the clustering pipeline of Posani, Wang,
Muscinelli, Paninski & Fusi (2026), imported from `Posani/clustering-analysis`
rather than reimplemented.

| panel | question | null |
|---|---|---|
| **a** | are HD tuning curves categorical? | Gaussian with matched mean and covariance |
| **b** | are a subject's neurons more clusterable than a random set? | size-matched pseudo-animals |
| **c** | do subjects cluster in shape space? | subject identity shuffled at the neuron level |
| **d** | is subject space categorical, or a continuum? | Gaussian with matched mean and covariance |

Panels a and b cluster **neurons** (each neuron is a point, described by its
tuning curve). Panels c and d cluster **subjects** (each subject is a point in
the space of Procrustes distances): c asks whether subject identity structures
that space at all, d whether the subjects in it form discrete groups. Every
result is cached, so re-running redraws the figure without recomputing. The
three-panel summary figure below is a, b and c; d is drawn on its own.

In [ ]:
import glob
from pathlib import Path

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import scipy.stats as sts
from scipy.io import loadmat
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import MDS
from sklearn.metrics import silhouette_score
from netrep.metrics import LinearMetric
import sys, tqdm

REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO / "Posani/clustering-analysis/single_area"))
from utils.clustering_analysis import cluster_analysis_condition_space  # noqa: E402
from utils.clustering_algo import clustering                            # noqa: E402
from utils.dr_algo import dr                                            # noqa: E402

OUT = REPO / "duszkiewicz_analyses/results/clustering"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

# house style of this dataset's figures (cf. inter_animal_variability.ipynb):
# 2.5 inch panels, no left spine, no y ticks, default font sizes
PANEL, NULLC, DATAC = 2.1, "0.75", "darkred"

SMOOTHING, MIN_NEURONS = 12, 40
N_NULL, R_PSEUDO, N_DRAW, N_PCS = 100, 20, 100, 20

## Data

`HdTuning_xval_moveEp.mat`, smoothing window 12, sessions with fewer than 40
neurons dropped -- the same neurons the shape-metric analyses use. Curves are
rolled to put each peak at the centre: preferred directions are near-uniform, so
without this the clustering would recover preferred angle rather than tuning
shape.

In [ ]:
sessions = sorted(glob.glob(str(REPO / "duszkiewicz_analyses/Dataset_1/*"))
                  + glob.glob(str(REPO / "duszkiewicz_analyses/Dataset_2/*")))
c1, c2, subject = [], [], []
for s in tqdm.tqdm(sessions, desc="loading"):
    hd = loadmat(s + "/Analysis/HdTuning_xval_moveEp.mat")
    a, b = hd["hAll1"][:, :, SMOOTHING], hd["hAll2"][:, :, SMOOTHING]
    if a.shape[1] < MIN_NEURONS:
        continue
    c1.append(a.T); c2.append(b.T)
    subject += [Path(s).name.split("-")[0]] * a.shape[1]

subject = np.array(subject)
X = np.concatenate([(u + v) / 2 for u, v in zip(c1, c2)])       # neurons x bins
X = np.stack([np.roll(x, X.shape[1] // 2 - int(np.argmax(x))) for x in X])
CM = X                                                          # panel c reuses it
subjects = np.unique(subject)
counts = np.array([np.sum(subject == s) for s in subjects])
print(f"{len(X)} neurons, {len(subjects)} subjects, {counts.min()}-{counts.max()} each")

## a. Are HD tuning curves categorical?

Posani's settings unchanged: k-means over k in [3, 20) with 50 restarts, against
100 draws from a multivariate Gaussian matched to the data's mean and
covariance. Gaussian rather than a shuffle on purpose -- it preserves the
covariance, so the test asks only whether the cloud is lumpy rather than
continuous.

In [ ]:
f = OUT / "posani_replication_posub.npz"
if f.exists():
    d = np.load(f)
    a_obs, a_null, a_z, a_k = (float(d["sscore_mean"]), d["sscore_nulls"],
                               float(d["sscore_z"]), int(d["k"]))
else:
    r = cluster_analysis_condition_space(
        X,
        {"min_N": 50, "algo": "kmeans", "n_init": 50, "n_clus_lim": [3, 20],
         "dis_metric": "euclidean", "save_id": "posub"},
        {"remove_sus_clus": False, "sessions_orig": np.zeros(len(X), int),
         "sus_clus_thres": 0.9},
        {"N_null": N_NULL, "null_dist": "Gaussian"},
        dict(plot=False, folder=str(OUT), save_id="posub"))
    a_obs, a_null = r["sscore_mean"], np.array(r["sscore_nulls"])
    a_z, a_k = r["sscore_z"], len(np.unique(r["clus_labels"]))
    np.savez(f, sscore_mean=a_obs, sscore_nulls=a_null, sscore_z=a_z, k=a_k)

print(f"a: silhouette {a_obs:.4f}  null {a_null.mean():.4f} +/- {a_null.std():.4f}"
      f"  z = {a_z:+.2f}  (k = {a_k})")

## b. Are a subject's neurons more clusterable than a random set?

Every subset is clustered in one PCA space fitted on all neurons, and k is capped
by sample size, so a real animal and its pseudo-animals are swept over an
identical range. Pseudo-animals are drawn without replacement from the whole
population -- a permutation of subject identity at fixed n, necessary because
silhouette falls with n and animals here have 42-185 neurons.

A sanity check (`pca_sanity.npz`) repeats this without PCA and at 95% and 99% of
variance: mean z moves by less than 0.05 and the per-animal values correlate at
r > 0.99, so the result does not depend on the projection.

In [ ]:
f = OUT / "within_vs_pseudo.csv"
if f.exists():
    b_res = pd.read_csv(f)
else:
    Xs = (X - X.mean(1, keepdims=True)) / X.std(1, keepdims=True)
    Z = dr(Xs, dict(method="pca", exp_var=0.9))
    rng = np.random.default_rng(0)

    def best_sil(Zsub):
        kmax = max(2, min(10, len(Zsub) // 10))
        return clustering(Zsub, "kmeans", n_clus_lim=[2, kmax + 1],
                          dis_metric="euclidean", n_init=50)["sscores_mean"]

    rows = []
    for s, n in tqdm.tqdm(list(zip(subjects, counts)), desc="subjects"):
        real = best_sil(Z[subject == s])
        null = np.array([best_sil(Z[rng.choice(len(Z), n, replace=False)])
                         for _ in range(R_PSEUDO)])
        rows.append(dict(subject=s, n=n, sil=real, null_mean=null.mean(),
                         z=(real - null.mean()) / (null.std() + 1e-12)))
    b_res = pd.DataFrame(rows)
    b_res.to_csv(f, index=False)

b_z = b_res.z.values
b_p = sts.wilcoxon(b_res.sil, b_res.null_mean).pvalue
print(f"b: mean z {b_z.mean():+.3f} (sd {b_z.std(ddof=1):.2f})  Wilcoxon p {b_p:.3f}"
      f"  {(b_z > 2).sum()} above +2, {(b_z < -2).sum()} below -2")

## c. Do subjects cluster in shape space?

Each subject becomes one point: all of its neurons, PCA to 20 components,
compared with the Procrustes distance. No cross-validation -- splitting a subject
in two exists so it can be compared with itself, which the identity question
needs and this one does not, and it would force groups of size two where
silhouette is biased hard negative.

The null permutes subject identity across **neurons** and rebuilds everything --
partitions, PCA, the full distance matrix. Permuting labels on a fixed distance
matrix would be close to tautological, since that matrix is built subject by
subject.

In [ ]:
def pair_dist(A, B, alpha):
    """From inter_animal_variability.ipynb."""
    m = LinearMetric(alpha=alpha, center_columns=True, score_method="euclidean")
    m.fit(A, B)
    return m.score(A, B)


def subject_distances(labels, alpha):
    pca = PCA(n_components=N_PCS)
    P = [pca.fit_transform(CM[labels == s].T) for s in np.unique(labels)]
    S = len(P); D = np.zeros((S, S))
    for i in range(S):
        for j in range(i + 1, S):
            D[i, j] = D[j, i] = pair_dist(P[i], P[j], alpha)
    return D


def cluster_strength(D, seed=0):
    """Posani's statistic: sweep k, keep the best silhouette. Their pipeline
    clusters coordinates, so the distances are embedded by MDS first."""
    emb = MDS(n_components=5, dissimilarity="precomputed", random_state=seed,
              normalized_stress=False, n_init=4).fit_transform(D)
    best = max(((silhouette_score(emb, lab), lab) for lab in
                (KMeans(n_clusters=k, n_init=50, init="random",
                        random_state=42).fit_predict(emb) for k in range(2, 8))),
               key=lambda t: t[0])
    return best[0], len(np.unique(best[1]))


f = OUT / "subject_clustering.npz"
if f.exists():
    z = np.load(f)
    c_obs, c_null = z["obs_a1"], z["null_a1"][:, 0]
else:
    rng = np.random.default_rng(0)
    c_obs = np.array(cluster_strength(subject_distances(subject, 1.0)))
    c_null = np.array([cluster_strength(
        subject_distances(subject[rng.permutation(len(subject))], 1.0), seed=d)[0]
        for d in tqdm.trange(N_DRAW, desc="null")])
    np.savez(f, obs_a1=c_obs, null_a1=np.c_[c_null, np.zeros_like(c_null)])
    c_null = c_null

c_p = (np.sum(c_null >= c_obs[0]) + 1) / (len(c_null) + 1)
c_zsc = (c_obs[0] - c_null.mean()) / c_null.std()
print(f"c: silhouette {c_obs[0]:.4f} (k = {int(c_obs[-1])})  "
      f"null {c_null.mean():.4f} +/- {c_null.std():.4f}  z = {c_zsc:+.2f}  p = {c_p:.3f}")

## d. Is subject space categorical, or a continuum?

Panel c asks whether subject identity structures the shape space at all; it does
not ask whether subjects fall into **discrete groups**. Silhouette cannot answer
that directly -- it needs at least two clusters, so it can never score the
one-cluster hypothesis, which has to be simulated instead.

Same test as panel a and as section 4 of `Posani/notebook.ipynb`, applied to
subjects rather than neurons or regions: embed the Procrustes distance matrix by
MDS, sweep k, keep the best silhouette, and compare it with draws from a
*single* Gaussian matched to that cloud's mean and covariance -- the real spread,
none of the lumpiness. The null makes the same free choice of k, so choosing k by
`argmax` costs nothing. With 31 subjects, k runs to 30 at most.

This is a different null from panel c's, and answers a different question. The
shuffle there destroys subject identity and rebuilds the distances; the Gaussian
here keeps the observed geometry and removes only its discreteness.

In [ ]:
def silhouette_sweep(E, ks, seed=42):
    """Best-of-k-means silhouette at each k (Posani's `shape.silhouette_sweep`)."""
    return np.array([silhouette_score(E, KMeans(n_clusters=k, n_init=50,
                     init="random", random_state=seed).fit_predict(E)) for k in ks])


def subject_embedding(D, seed=0):
    """The same MDS space `cluster_strength` clusters in, kept rather than discarded."""
    return MDS(n_components=5, dissimilarity="precomputed", random_state=seed,
               normalized_stress=False, n_init=4).fit_transform(D)


KS = np.arange(2, len(subjects))          # 2..30, the maximum for 31 subjects

f = OUT / "subject_gaussian_null.npz"
if f.exists():
    z = np.load(f)
    d_sweep, d_null_sweep = z["obs_sweep"], z["null_sweep"]
else:
    E_sub = subject_embedding(subject_distances(subject, 1.0))
    d_sweep = silhouette_sweep(E_sub, KS)
    # one continuous cloud with the real mean and covariance, swept identically
    rng = np.random.default_rng(0)
    mu, S = E_sub.mean(0), np.cov(E_sub.T)
    d_null_sweep = np.array([silhouette_sweep(rng.multivariate_normal(mu, S, len(E_sub)), KS)
                             for _ in tqdm.trange(N_DRAW, desc="gaussian null")])
    np.savez(f, obs_sweep=d_sweep, null_sweep=d_null_sweep, emb=E_sub)

d_obs, d_k = d_sweep.max(), KS[d_sweep.argmax()]
d_null = d_null_sweep.max(1)                       # the null picks its own best k too
d_zsc = (d_obs - d_null.mean()) / d_null.std()
d_p = (np.sum(d_null >= d_obs) + 1) / (len(d_null) + 1)
print(f"d: silhouette {d_obs:.4f} (k = {d_k})  Gaussian null {d_null.mean():.4f}"
      f" +/- {d_null.std():.4f}  z = {d_zsc:+.2f}  p = {d_p:.3f}")
print(f"   the null picks k = 2 in {np.mean(KS[d_null_sweep.argmax(1)] == 2):.0%} of its draws")

fig, ax = plt.subplots(figsize=(PANEL, PANEL))
ax.hist(d_null, bins=20, color=NULLC, label="one continuous cloud")
ax.axvline(d_obs, color=DATAC, lw=2, label="real subjects")
ax.set_xlabel("best silhouette over k")
ax.set_yticks([])
lo, hi = ax.get_ylim(); ax.set_ylim(lo, hi * 1.22)
ax.legend(frameon=False, fontsize=7, loc="upper left", handlelength=1.0)
ax.annotate(f"z = {d_zsc:+.2f}, p = {d_p:.3f}", (.04, .78), xycoords="axes fraction",
            fontsize=7, va="top")
ax.set_box_aspect(1)
sns.despine(ax=ax, left=True)
fig.tight_layout()
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"subject_gaussian_null.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

## The figure

Two-column width, three panels, house style throughout: no left spine, no y
ticks, default font sizes. Grey is always the null, dark red always the data.

In [ ]:
def style(ax, xlabel, loc):
    """House style, plus headroom so the legend clears the data."""
    ax.set_xlabel(xlabel)
    ax.set_yticks([])
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, hi * 1.22)
    ax.legend(frameon=False, fontsize=7, loc=loc, handlelength=1.0)
    ax.set_box_aspect(1)          # square plot area, so the row stays compact
    sns.despine(ax=ax, left=True)


fig, axes = plt.subplots(1, 3, figsize=(3 * PANEL, PANEL))

ax = axes[0]
ax.hist(a_null, bins=20, color=NULLC, label="null model")
ax.axvline(a_obs, color=DATAC, lw=2, label="data")
style(ax, "mean silhouette", "upper left")

ax = axes[1]
ax.hist(b_z, bins=np.arange(np.floor(b_z.min()) - .5, np.ceil(b_z.max()) + 1, .5),
        color=NULLC, label="subjects")
ax.axvline(0, color="0.45", lw=1, ls="--")          # shuffle control, at z = 0
ax.axvline(b_z.mean(), color=DATAC, lw=2, label=f"mean {b_z.mean():+.2f}")
style(ax, "clustering strength (z)", "upper right")

ax = axes[2]
ax.hist(c_null, bins=20, color=NULLC, label="shuffled subjects")
ax.axvline(c_obs[0], color=DATAC, lw=2, label="real subjects")
style(ax, "best silhouette", "upper left")

fig.tight_layout()
for e in ("svg", "pdf", "png"):
    fig.savefig(OUT / f"hd_clustering_panels.{e}", bbox_inches="tight",
                dpi=300 if e == "png" else None)
plt.show()

# the statistics the titles used to carry, for the caption
print(f"a  silhouette {a_obs:.4f} vs null {a_null.mean():.4f} +/- {a_null.std():.4f}"
      f"   z = {a_z:+.2f}   k = {a_k}")
print(f"b  mean z {b_z.mean():+.3f} (sd {b_z.std(ddof=1):.2f})   Wilcoxon p = {b_p:.3f}"
      f"   {(b_z > 2).sum()} above +2, {(b_z < -2).sum()} below -2")
print(f"c  silhouette {c_obs[0]:.4f} vs null {c_null.mean():.4f} +/- {c_null.std():.4f}"
      f"   z = {c_zsc:+.2f}   p = {c_p:.3f}   k = {int(c_obs[-1])}")

Two caveats for the text. Subject and session are one-to-one in this dataset, so
"subject-specific" and "recording-specific" cannot be separated -- panel c's
positive result carries that ambiguity. And panel b's null uses 20 draws per
subject, so the smallest attainable per-subject p is 1/21 = 0.048; the few
subjects near that floor are unresolved rather than significant.

Panels c and d together: subject identity **does** structure the shape space
(c, silhouette 0.436 against shuffled subjects at 0.359 +/- 0.033, z = +2.33,
p = 0.020), but the subjects in it are **not** categorical (d, the same 0.436
against a continuous cloud's own best of 0.405 +/- 0.052, z = +0.58, p = 0.27).
The same picture as the cortical regions in `Posani/notebook.ipynb`: a space that
is organised, and continuous. Note also that the Gaussian null picks k = 2 in 92%
of its own draws -- k-means will halve any elongated cloud -- so the data's
winning k = 2 is weak evidence by itself.

The two panels share an observed value by construction: c and d score the same
embedding with the same statistic and differ only in the null, so the single
number 0.436 is read against a shuffle in c and against a Gaussian in d.
